# Task 1 — Baseline MLP
**The Concept:** Flatten the 2D MFCC feature map (Time × Frequency) into a 1D array and pass it through standard Dense layers.

**Pipeline:**
```
Raw Audio → MFCC (51 × 13 × 1) → Flatten → Dense(32) → Dense(16) → Softmax(2)
```

## Setup

In [ ]:
# Install dependencies (first run only)
# !pip install tensorflow tensorflow_datasets librosa matplotlib scikit-learn

import tensorflow as tf
import tensorflow_datasets as tfds
import numpy as np
import matplotlib.pyplot as plt
import librosa
import os
import sys
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight
from tensorflow.keras import layers, models

plt.style.use('seaborn-v0_8-whitegrid')
tf.get_logger().setLevel('ERROR')

print(f'TensorFlow: {tf.__version__}')
print(f'Python:     {sys.version.split()[0]}')
print(f'Librosa:    {librosa.__version__}')

## Load Google Speech Commands v2 (Yes/No Only)

In [ ]:
def load_speech_commands_tfds(batch_size=32, sample_rate=16000, max_duration=1.0):
    """Load Yes/No/Silence samples from Speech Commands v0.0.3 via TFDS."""
    ds, info = tfds.load(
        'speech_commands:0.0.3',
        split='train',
        with_info=True,
        shuffle_files=True
    )

    # Filter: keep only silence(0), yes(2), no(3)
    def filter_yes_no_silence(example):
        label = example['label']
        return tf.logical_or(
            tf.logical_or(label == 0, label == 2),
            label == 3
        )

    ds_filtered = ds.filter(filter_yes_no_silence)

    def preprocess(example):
        audio = example['audio']
        label = example['label']
        audio = tf.cast(audio, tf.float32) / 32768.0
        target_length = int(sample_rate * max_duration)
        if tf.shape(audio)[0] > target_length:
            audio = audio[:target_length]
        else:
            audio = tf.pad(audio, [[0, target_length - tf.shape(audio)[0]]])
        # Binary: yes=1, no/silence=0
        label_binary = tf.cond(
            label == 2,
            lambda: tf.constant(1, dtype=tf.int32),
            lambda: tf.constant(0, dtype=tf.int32)
        )
        return audio, label_binary

    ds_processed = ds_filtered.map(preprocess, num_parallel_calls=tf.data.AUTOTUNE)

    audios_list, labels_list = [], []
    for audio, label in ds_processed.as_numpy_iterator():
        audios_list.append(audio)
        labels_list.append(label)

    X = np.array(audios_list, dtype=np.float32)
    y = np.array(labels_list, dtype=np.int32)
    return X, y

print('Loading Google Speech Commands v2 (Yes/No subset) via TFDS...')
X, y = load_speech_commands_tfds()

idx = np.random.permutation(len(X))
X, y = X[idx], y[idx]

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Loaded: {len(X)} samples | Yes: {np.sum(y==1)}, No: {np.sum(y==0)}')
print(f'Train: {len(X_train)} | Val: {len(X_val)}')

## Visualize Class Distribution & Sample Waveforms

In [ ]:
import random

print('Dataset Summary:')
print(f'  Total samples : {len(y)}')
print(f'  Yes (1)       : {np.sum(y==1)} ({np.mean(y==1)*100:.1f}%)')
print(f'  No/Silence (0): {np.sum(y==0)} ({np.mean(y==0)*100:.1f}%)')
print(f'  Imbalance     : {np.sum(y==0)/np.sum(y==1):.2f}x')

colors = ['#ff6b6b', '#4ecdc4']

# Class distribution bar chart
plt.figure(figsize=(6, 4))
labels_names = ['No/Silence (0)', 'Yes (1)']
counts = [np.sum(y==0), np.sum(y==1)]
plt.bar(labels_names, counts, color=colors, edgecolor='black', alpha=0.9)
plt.title('Class Distribution: Yes vs No/Silence')
plt.ylabel('Number of Samples')
for i, v in enumerate(counts):
    plt.text(i, v + 100, str(v), ha='center', fontweight='bold')
plt.grid(axis='y', alpha=0.3, linestyle='--')
plt.tight_layout(); plt.show()

# Sample waveforms
fig, axes = plt.subplots(2, 2, figsize=(12, 6))
axes = axes.flatten()
yes_idx = np.where(y == 1)[0]
no_idx  = np.where(y == 0)[0]
sample_indices = random.sample(list(yes_idx), 2) + random.sample(list(no_idx), 2)
random.shuffle(sample_indices)
for i, idx in enumerate(sample_indices):
    audio = X[idx]
    label = 'Yes' if y[idx] == 1 else 'No/Silence'
    t = np.linspace(0, 1, len(audio))
    axes[i].plot(t, audio, linewidth=0.5, color=colors[y[idx]])
    axes[i].set_title(f'Sample #{idx+1} | {label}')
    axes[i].set_xlabel('Time (s)'); axes[i].set_ylabel('Amplitude')
    axes[i].set_ylim(-1.2, 1.2); axes[i].grid(alpha=0.3)
plt.tight_layout(); plt.show()

## Extract & Normalize MFCC Features

Using **librosa** with `n_mfcc=13`, `n_fft=512`, `hop_length=320` → shape `(51, 13, 1)`.

In [ ]:
def audio_to_mfcc(audio, sr=16000, n_mfcc=13):
    mfcc = librosa.feature.mfcc(
        y=audio, sr=sr, n_mfcc=n_mfcc, n_fft=512, hop_length=320
    )
    mfcc = mfcc.T                          # (frames, n_mfcc)
    return np.expand_dims(mfcc, axis=-1)   # (51, 13, 1)

print('Extracting MFCC features...')
X_train_mfcc = np.array([audio_to_mfcc(a) for a in X_train])
X_val_mfcc   = np.array([audio_to_mfcc(a) for a in X_val])

# Normalize with StandardScaler
scaler = StandardScaler()
N_train, T, F, C = X_train_mfcc.shape
X_train_mfcc = scaler.fit_transform(X_train_mfcc.reshape(N_train, -1)).reshape(N_train, T, F, C)
N_val = X_val_mfcc.shape[0]
X_val_mfcc = scaler.transform(X_val_mfcc.reshape(N_val, -1)).reshape(N_val, T, F, C)

INPUT_SHAPE = X_train_mfcc.shape[1:]   # (51, 13, 1)
NUM_CLASSES = 2

print(f'Train: {X_train_mfcc.shape} | Val: {X_val_mfcc.shape}')
print(f'Model input shape: {INPUT_SHAPE}')

# MFCC visualization for one Yes and one No sample
def plot_mfcc(audio, title, cmap='viridis'):
    mfcc = librosa.feature.mfcc(y=audio, sr=16000, n_mfcc=13, n_fft=512, hop_length=320)
    plt.figure(figsize=(8, 3))
    plt.imshow(mfcc, aspect='auto', origin='lower', cmap=cmap)
    plt.title(title); plt.xlabel('Time Frame'); plt.ylabel('MFCC Coefficient')
    plt.colorbar(fraction=0.046, pad=0.04); plt.tight_layout(); plt.show()

plot_mfcc(X[np.where(y==1)[0][0]], "MFCC: 'Yes' Keyword", cmap='plasma')
plot_mfcc(X[np.where(y==0)[0][0]], "MFCC: 'No' Keyword",  cmap='viridis')

# Class weights for imbalance
class_weights = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
class_weight_dict = dict(enumerate(class_weights))
print(f'Class weights: {class_weight_dict}')

## Build Baseline MLP Model

Flatten `(51 × 13 × 1)` → `663` inputs → Dense(32) → Dense(16) → Softmax(2)

In [ ]:
def build_baseline_mlp(input_shape, num_classes):
    model = models.Sequential([
        layers.Input(shape=input_shape, name='mfcc_input'),
        layers.Flatten(name='flatten_mfcc'),
        layers.Dense(32, activation='relu', name='dense_1'),
        layers.Dense(16, activation='relu', name='dense_2'),
        layers.Dense(num_classes, activation='softmax', name='output')
    ], name='Baseline_MLP')
    return model

mlp_model = build_baseline_mlp(INPUT_SHAPE, NUM_CLASSES)
mlp_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)
mlp_model.summary()

## Train Baseline MLP

In [ ]:
import time

callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, min_lr=1e-5)
]

print('Starting training...')
t0 = time.time()
history = mlp_model.fit(
    X_train_mfcc, y_train,
    validation_data=(X_val_mfcc, y_val),
    epochs=15,
    batch_size=32,
    class_weight=class_weight_dict,
    callbacks=callbacks,
    verbose=1
)
time_per_epoch = (time.time() - t0) / len(history.history['loss'])

# Training curves
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'],     label='Train Acc', marker='o')
plt.plot(history.history['val_accuracy'], label='Val Acc',   marker='s')
plt.title('Accuracy'); plt.legend(); plt.grid()
plt.subplot(1, 2, 2)
plt.plot(history.history['loss'],     label='Train Loss', marker='o')
plt.plot(history.history['val_loss'], label='Val Loss',   marker='s')
plt.title('Loss'); plt.legend(); plt.grid()
plt.suptitle('Task 1 — Baseline MLP', fontsize=13)
plt.tight_layout(); plt.show()

print(f'Final Val Accuracy : {max(history.history["val_accuracy"]):.3f}')
print(f'Avg Time / Epoch   : {time_per_epoch:.1f} s')

## Upgrade 1 — Add Dropout Regularization

In [ ]:
def build_mlp_dropout(input_shape, num_classes):
    return tf.keras.Sequential([
        layers.Input(shape=input_shape, name='mfcc_input'),
        layers.Flatten(name='flatten'),
        layers.Dense(32, activation='relu', name='dense_1'),
        layers.Dropout(0.3, name='drop_1'),
        layers.Dense(16, activation='relu', name='dense_2'),
        layers.Dropout(0.3, name='drop_2'),
        layers.Dense(num_classes, activation='softmax', name='output')
    ], name='MLP_Dropout')

print('Building MLP + Dropout...')
model_drop = build_mlp_dropout(INPUT_SHAPE, NUM_CLASSES)
model_drop.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

print('Training...')
history_drop = model_drop.fit(
    X_train_mfcc, y_train,
    validation_data=(X_val_mfcc, y_val),
    epochs=20,
    batch_size=32,
    class_weight=class_weight_dict,
    callbacks=[tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=4, restore_best_weights=True)],
    verbose=1
)

plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.plot(history_drop.history['accuracy'],     label='Train Acc', marker='o', linewidth=2)
plt.plot(history_drop.history['val_accuracy'], label='Val Acc',   marker='s', linewidth=2)
plt.title('Dropout MLP — Accuracy'); plt.legend(); plt.grid(alpha=0.3)
plt.subplot(1, 2, 2)
plt.plot(history_drop.history['loss'],     label='Train Loss', marker='o', linewidth=2)
plt.plot(history_drop.history['val_loss'], label='Val Loss',   marker='s', linewidth=2)
plt.title('Dropout MLP — Loss'); plt.legend(); plt.grid(alpha=0.3)
plt.tight_layout(); plt.show()

print(f'Dropout Val Accuracy: {max(history_drop.history["val_accuracy"]):.3f}')
print(f'Params: {model_drop.count_params():,}')

## Export Best Model to TFLite (int8)

In [ ]:
def representative_dataset():
    for i in range(min(100, len(X_train_mfcc))):
        yield [X_train_mfcc[i:i+1].astype(np.float32)]

print('Exporting Baseline MLP to TFLite...')
converter = tf.lite.TFLiteConverter.from_keras_model(mlp_model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset = representative_dataset
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type  = tf.float32
converter.inference_output_type = tf.float32

tflite_model = converter.convert()
model_path = 'kws_yesno_baseline_mlp.tflite'
with open(model_path, 'wb') as f:
    f.write(tflite_model)

print(f'Saved: {model_path}')
print(f'Size : {os.path.getsize(model_path)/1024:.2f} KB')

# ---- Summary for benchmark table ----
print('\n' + '='*50)
print('TASK 1 — BASELINE MLP RESULTS (for benchmark)')
print('='*50)
print(f'  Architecture   : Baseline MLP')
print(f'  Total Params   : {mlp_model.count_params():,}')
print(f'  Val Accuracy   : {max(history.history["val_accuracy"])*100:.2f}%')
print(f'  Time/Epoch     : {time_per_epoch:.1f} s')
print('='*50)